# Single Notebook CNN Investigation (iCoSimal V3)

This notebook is organized directly by the **mandatory objectives (a–e)** and includes **initial data analysis**.

## Mandatory objectives checklist
- **(a)** Start from simple CNN architectures and progressively increase complexity/depth.
- **(b)** Show the importance of hyperparameter tuning.
- **(c)** Show underfitting and overfitting cases and explain reasons.
- **(d)** Experiment with regularization techniques.
- **(e)** Compare optimization algorithms (Adam, RMSprop, SGD).

> Practical tip: begin with smaller image sizes (64×64 / 128×128) for quicker experimentation.


## 0) Setup


In [ ]:
# If needed, install dependencies first:
# !pip install -r ../requirements.txt

import warnings
from itertools import product
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix


# Notebook -> repository root
REPO_ROOT = Path('..').resolve()

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


# ---------------------------------------------------------------------------
# Inlined helpers (mirroring src/* so notebook can run standalone)
# ---------------------------------------------------------------------------

def get_transforms(image_size: int = 224, augment: bool = True):
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )

    if augment:
        train_transform = transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(
                    brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1
                ),
                transforms.ToTensor(),
                normalize,
            ]
        )
    else:
        train_transform = transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                normalize,
            ]
        )

    val_transform = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            normalize,
        ]
    )

    return train_transform, val_transform


def get_dataloaders(
    data_root: str,
    image_size: int = 224,
    batch_size: int = 32,
    num_workers: int = 4,
    augment: bool = True,
):
    data_root = Path(data_root)
    train_dir = data_root / 'train'
    val_dir = data_root / 'validate'

    if not train_dir.exists():
        raise FileNotFoundError(
            f"Training directory not found: {train_dir}\n"
            "Please set data_root to the directory containing 'train/' and 'validate/'."
        )
    if not val_dir.exists():
        raise FileNotFoundError(
            f"Validation directory not found: {val_dir}\n"
            "Please set data_root to the directory containing 'train/' and 'validate/'."
        )

    train_transform, val_transform = get_transforms(image_size=image_size, augment=augment)
    train_dataset = datasets.ImageFolder(root=str(train_dir), transform=train_transform)
    val_dataset = datasets.ImageFolder(root=str(val_dir), transform=val_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    class_names = train_dataset.classes
    print(f'Classes found: {class_names}')
    print(f'Dataset sizes  — train: {len(train_dataset)}, validation: {len(val_dataset)}')

    return train_loader, val_loader, class_names


class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10, input_size: int = 64, dropout: float = 0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        flat_size = 64 * (input_size // 4) * (input_size // 4)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)


class MediumCNN(nn.Module):
    def __init__(self, num_classes: int = 10, input_size: int = 64, dropout: float = 0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        flat_size = 256 * (input_size // 16) * (input_size // 16)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)


class DeepCNN(nn.Module):
    def __init__(self, num_classes: int = 10, input_size: int = 64, dropout: float = 0.5):
        super().__init__()

        def conv_block(in_ch: int, out_ch: int) -> nn.Sequential:
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            )

        self.features = nn.Sequential(
            conv_block(3, 32),
            conv_block(32, 32),
            nn.MaxPool2d(kernel_size=2, stride=2),
            conv_block(32, 64),
            conv_block(64, 64),
            nn.MaxPool2d(kernel_size=2, stride=2),
            conv_block(64, 128),
            conv_block(128, 128),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


def get_model(
    architecture: str,
    num_classes: int = 10,
    input_size: int = 64,
    dropout: float = 0.5,
) -> nn.Module:
    architectures = {
        'simple': SimpleCNN,
        'medium': MediumCNN,
        'deep': DeepCNN,
    }
    if architecture not in architectures:
        raise ValueError(
            f"Unknown architecture '{architecture}'. Choose from {list(architectures.keys())}."
        )
    return architectures[architecture](
        num_classes=num_classes,
        input_size=input_size,
        dropout=dropout,
    )


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> Tuple[float, float]:
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in tqdm(loader, desc='  Train', leave=False):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(targets).sum().item()
        total += inputs.size(0)

    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in tqdm(loader, desc='  Val  ', leave=False):
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(targets).sum().item()
        total += inputs.size(0)

    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy


def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    *,
    num_epochs: int = 20,
    learning_rate: float = 1e-3,
    optimizer_name: str = 'adam',
    weight_decay: float = 1e-4,
    scheduler_name: str = 'step',
    device: torch.device | None = None,
    verbose: bool = True,
) -> Dict[str, List[float]]:
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name.lower() == 'adam':
        optimizer = torch.optim.Adam(
            model.parameters(), lr=learning_rate, weight_decay=weight_decay
        )
    elif optimizer_name.lower() == 'rmsprop':
        optimizer = torch.optim.RMSprop(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
            momentum=0.9,
        )
    elif optimizer_name.lower() == 'sgd':
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=learning_rate,
            momentum=0.9,
            weight_decay=weight_decay,
        )
    else:
        raise ValueError(
            f"Unknown optimizer '{optimizer_name}'. Choose 'adam', 'rmsprop', or 'sgd'."
        )

    if scheduler_name.lower() == 'step':
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    elif scheduler_name.lower() == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    else:
        scheduler = None

    history: Dict[str, List[float]] = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
    }

    best_val_acc = 0.0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        if scheduler is not None:
            scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        if verbose:
            print(
                f"Epoch [{epoch:03d}/{num_epochs}]  "
                f"train_loss={train_loss:.4f}  train_acc={train_acc:.2f}%  "
                f"val_loss={val_loss:.4f}  val_acc={val_acc:.2f}%"
            )

    if verbose:
        print(f'\nBest validation accuracy: {best_val_acc:.2f}%')

    return history


@torch.no_grad()
def predict(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_targets: List[int] = []
    all_preds: List[int] = []

    for inputs, targets in tqdm(loader, desc='Predicting'):
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        all_targets.extend(targets.numpy().tolist())
        all_preds.extend(predicted.cpu().numpy().tolist())

    return np.array(all_targets), np.array(all_preds)


def print_classification_report(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: List[str],
) -> None:
    print('\n=== Classification Report ===')
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))


def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: List[str],
    title: str = 'Confusion Matrix',
    figsize: Tuple[int, int] = (10, 8),
) -> plt.Figure:
    cm = confusion_matrix(y_true, y_pred)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.zeros_like(cm, dtype=float)
    np.divide(cm.astype(float), row_sums, out=cm_norm, where=row_sums != 0)

    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title(title)
    plt.tight_layout()
    return fig


def plot_training_history(
    history: dict,
    title: str = 'Training History',
    figsize: Tuple[int, int] = (12, 5),
) -> plt.Figure:
    epochs = range(1, len(history['train_loss']) + 1)

    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=figsize)
    fig.suptitle(title)

    ax_loss.plot(epochs, history['train_loss'], label='Train loss')
    ax_loss.plot(epochs, history['val_loss'], label='Val loss')
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('Cross-entropy loss')
    ax_loss.legend()
    ax_loss.grid(True)

    ax_acc.plot(epochs, history['train_acc'], label='Train acc')
    ax_acc.plot(epochs, history['val_acc'], label='Val acc')
    ax_acc.set_xlabel('Epoch')
    ax_acc.set_ylabel('Accuracy (%)')
    ax_acc.legend()
    ax_acc.grid(True)

    plt.tight_layout()
    return fig


def compare_architectures(
    results: dict,
    metric: str = 'val_acc',
    figsize: Tuple[int, int] = (10, 6),
) -> plt.Figure:
    fig, ax = plt.subplots(figsize=figsize)

    for label, history in results.items():
        epochs = range(1, len(history[metric]) + 1)
        ax.plot(epochs, history[metric], marker='o', markersize=3, label=label)

    metric_label = metric.replace('_', ' ').title()
    ax.set_xlabel('Epoch')
    ax.set_ylabel(metric_label)
    ax.set_title(f'Architecture Comparison — {metric_label}')
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    return fig


In [ ]:
# Set this to your local dataset path (must contain train/ and validate/)
DATA_ROOT = '/path/to/icosimal_img_class_03/data_uniform_224_224_sets'

# Global quick-run defaults (increase later for stronger experiments)
DEFAULT_IMAGE_SIZE = 64
DEFAULT_BATCH_SIZE = 32
DEFAULT_EPOCHS = 8
NUM_WORKERS = 4


## 1) Initial Data Analysis


In [ ]:
# Basic dataset existence checks
train_dir = Path(DATA_ROOT) / 'train'
val_dir = Path(DATA_ROOT) / 'validate'
print('Train dir:', train_dir)
print('Val dir:  ', val_dir)
print('Train exists?', train_dir.exists())
print('Val exists?  ', val_dir.exists())

if not train_dir.exists() or not val_dir.exists():
    raise FileNotFoundError('Please set DATA_ROOT to the extracted dataset folder with train/ and validate/.')


In [ ]:
# Class counts and split summary
class_names = sorted([p.name for p in train_dir.iterdir() if p.is_dir()])

train_counts = {c: len(list((train_dir / c).glob('*'))) for c in class_names}
val_counts = {c: len(list((val_dir / c).glob('*'))) for c in class_names}

eda_df = pd.DataFrame({
    'class': class_names,
    'train_count': [train_counts[c] for c in class_names],
    'val_count': [val_counts[c] for c in class_names],
})
eda_df['total'] = eda_df['train_count'] + eda_df['val_count']

display(eda_df)
print('Total train images:', eda_df['train_count'].sum())
print('Total val images:  ', eda_df['val_count'].sum())


In [ ]:
# Plot class distribution for train/validation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=eda_df, x='class', y='train_count', ax=axes[0], color='steelblue')
axes[0].set_title('Train split class counts')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(data=eda_df, x='class', y='val_count', ax=axes[1], color='darkorange')
axes[1].set_title('Validation split class counts')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Sample raw image-size statistics (before transforms)
def sample_image_sizes(root_dir, max_per_class=200):
    sizes = []
    for cls_dir in sorted([p for p in Path(root_dir).iterdir() if p.is_dir()]):
        for i, img_path in enumerate(cls_dir.glob('*')):
            if i >= max_per_class:
                break
            try:
                with Image.open(img_path) as img:
                    w, h = img.size
                sizes.append((cls_dir.name, w, h))
            except Exception:
                continue
    return pd.DataFrame(sizes, columns=['class', 'width', 'height'])

size_df = sample_image_sizes(train_dir)
display(size_df.describe(include='all'))
\n

In [ ]:
# Visualize a mini-batch after transforms
train_loader, val_loader, CLASS_NAMES = get_dataloaders(
    data_root=DATA_ROOT,
    image_size=DEFAULT_IMAGE_SIZE,
    batch_size=DEFAULT_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    augment=True,
)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flatten()):
    img = images[i].permute(1, 2, 0).cpu().numpy()
    # de-normalize roughly for display
    img = np.clip((img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406]), 0, 1)
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[labels[i].item()])
    ax.axis('off')
plt.tight_layout()
plt.show()


## 2) Objective (a): Progressive Depth (Simple → Medium → Deep)

We keep training settings fixed and only change architecture depth/complexity.


In [ ]:
DEPTH_CFG = dict(
    image_size=64,
    batch_size=32,
    num_epochs=8,
    learning_rate=1e-3,
    optimizer_name='adam',
    weight_decay=1e-4,
    dropout=0.5,
    scheduler_name='cosine',
    augment=True,
)

train_loader, val_loader, CLASS_NAMES = get_dataloaders(
    DATA_ROOT,
    image_size=DEPTH_CFG['image_size'],
    batch_size=DEPTH_CFG['batch_size'],
    num_workers=NUM_WORKERS,
    augment=DEPTH_CFG['augment'],
)

arch_histories = {}
arch_models = {}
for arch in ['simple', 'medium', 'deep']:
    print(f"\n=== Training architecture: {arch.upper()} ===")
    model = get_model(
        architecture=arch,
        num_classes=len(CLASS_NAMES),
        input_size=DEPTH_CFG['image_size'],
        dropout=DEPTH_CFG['dropout'],
    )
    print('Trainable params:', f"{count_parameters(model):,}")

    history = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=DEPTH_CFG['num_epochs'],
        learning_rate=DEPTH_CFG['learning_rate'],
        optimizer_name=DEPTH_CFG['optimizer_name'],
        weight_decay=DEPTH_CFG['weight_decay'],
        scheduler_name=DEPTH_CFG['scheduler_name'],
        device=DEVICE,
        verbose=True,
    )
    arch_histories[arch] = history
    arch_models[arch] = model


In [ ]:
# Compare validation accuracy curves for objective (a)
_ = compare_architectures(arch_histories, metric='val_acc', figsize=(9, 5))
plt.show()

depth_summary = pd.DataFrame([
    {
        'architecture': arch,
        'best_val_acc': max(hist['val_acc']),
        'final_train_acc': hist['train_acc'][-1],
        'params': count_parameters(arch_models[arch]),
    }
    for arch, hist in arch_histories.items()
]).sort_values('best_val_acc', ascending=False)

display(depth_summary)


### Objective (a) Interpretation
- As depth increases, the model has greater representational power and usually reaches higher validation accuracy.
- The deep model also benefits from BatchNorm and a stronger feature hierarchy.
- If deeper models do not outperform shallower ones, likely causes are insufficient epochs, overly small learning rate, or regularization settings.


## 3) Objective (b): Hyperparameter Tuning Importance

We run a compact sweep first (fast) and identify which settings move validation accuracy the most.


In [ ]:
hparam_grid = {
    'image_size': [64, 128],
    'batch_size': [32, 64],
    'learning_rate': [1e-3, 3e-4, 1e-4],
    'optimizer_name': ['adam'],
    'weight_decay': [1e-4],
    'dropout': [0.3, 0.5],
    'scheduler_name': ['cosine'],
    'architecture': ['medium'],
    'num_epochs': [6],
}

keys = list(hparam_grid.keys())
configs = [dict(zip(keys, vals)) for vals in product(*[hparam_grid[k] for k in keys])]
print('Total sweep configurations:', len(configs))

sweep_records = []
for i, cfg in enumerate(configs, 1):
    print(f"\n[{i}/{len(configs)}] {cfg}")
    train_loader, val_loader, _ = get_dataloaders(
        DATA_ROOT,
        image_size=cfg['image_size'],
        batch_size=cfg['batch_size'],
        num_workers=NUM_WORKERS,
        augment=True,
    )
    model = get_model(
        architecture=cfg['architecture'],
        num_classes=10,
        input_size=cfg['image_size'],
        dropout=cfg['dropout'],
    )
    history = train(
        model,
        train_loader,
        val_loader,
        num_epochs=cfg['num_epochs'],
        learning_rate=cfg['learning_rate'],
        optimizer_name=cfg['optimizer_name'],
        weight_decay=cfg['weight_decay'],
        scheduler_name=cfg['scheduler_name'],
        device=DEVICE,
        verbose=False,
    )
    sweep_records.append({
        **cfg,
        'best_val_acc': max(history['val_acc']),
        'final_val_acc': history['val_acc'][-1],
        'final_train_acc': history['train_acc'][-1],
    })

sweep_df = pd.DataFrame(sweep_records).sort_values('best_val_acc', ascending=False)
display(sweep_df.head(10))


In [ ]:
# Visualize tuning effects
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=sweep_df, x='image_size', y='best_val_acc', ax=axes[0])
axes[0].set_title('Impact of image_size')

sns.boxplot(data=sweep_df, x='batch_size', y='best_val_acc', ax=axes[1])
axes[1].set_title('Impact of batch_size')

lr_effect = sweep_df.groupby('learning_rate', as_index=False)['best_val_acc'].mean().sort_values('learning_rate')
axes[2].plot(lr_effect['learning_rate'], lr_effect['best_val_acc'], marker='o')
axes[2].set_xscale('log')
axes[2].set_title('Impact of learning_rate (mean best val acc, log scale)')

plt.tight_layout()
plt.show()
\n

### Objective (b) Interpretation
- Small hyperparameter choices (especially learning rate and image size) can noticeably change validation performance.
- Tuning can produce bigger gains than architecture swaps under fixed training budgets.
- Start with small image size and fewer epochs for fast iteration, then scale up promising configurations.


## 4) Objective (c): Underfitting vs Overfitting

We intentionally configure one model to underfit and one to overfit, then compare learning curves.


In [ ]:
# --- Underfitting setup ---
underfit_train_loader, underfit_val_loader, _ = get_dataloaders(
    DATA_ROOT,
    image_size=64,
    batch_size=64,
    num_workers=NUM_WORKERS,
    augment=True,
)
underfit_model = get_model('simple', num_classes=10, input_size=64, dropout=0.7)
underfit_history = train(
    underfit_model,
    underfit_train_loader,
    underfit_val_loader,
    num_epochs=4,
    learning_rate=1e-4,
    optimizer_name='sgd',
    weight_decay=1e-2,
    scheduler_name='none',
    device=DEVICE,
    verbose=True,
)

# --- Overfitting setup (small subset + weak regularization) ---
train_tf, val_tf = get_transforms(image_size=64, augment=False)
full_train_dataset = datasets.ImageFolder(str(train_dir), transform=train_tf)
small_indices = list(range(min(1000, len(full_train_dataset))))
small_train_dataset = Subset(full_train_dataset, small_indices)

small_train_loader = DataLoader(
    small_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
_, overfit_val_loader, _ = get_dataloaders(
    DATA_ROOT,
    image_size=64,
    batch_size=32,
    num_workers=NUM_WORKERS,
    augment=False,
)
overfit_model = get_model('deep', num_classes=10, input_size=64, dropout=0.0)
overfit_history = train(
    overfit_model,
    small_train_loader,
    overfit_val_loader,
    num_epochs=12,
    learning_rate=1e-3,
    optimizer_name='adam',
    weight_decay=0.0,
    scheduler_name='none',
    device=DEVICE,
    verbose=True,
)


In [ ]:
# Plot underfit vs overfit histories
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Underfit
axes[0, 0].plot(underfit_history['train_loss'], label='train')
axes[0, 0].plot(underfit_history['val_loss'], label='val')
axes[0, 0].set_title('Underfitting: Loss')
axes[0, 0].legend()

axes[0, 1].plot(underfit_history['train_acc'], label='train')
axes[0, 1].plot(underfit_history['val_acc'], label='val')
axes[0, 1].set_title('Underfitting: Accuracy')
axes[0, 1].legend()

# Overfit
axes[1, 0].plot(overfit_history['train_loss'], label='train')
axes[1, 0].plot(overfit_history['val_loss'], label='val')
axes[1, 0].set_title('Overfitting: Loss')
axes[1, 0].legend()

axes[1, 1].plot(overfit_history['train_acc'], label='train')
axes[1, 1].plot(overfit_history['val_acc'], label='val')
axes[1, 1].set_title('Overfitting: Accuracy')
axes[1, 1].legend()

plt.tight_layout()
plt.show()


### Objective (c) Explanation
- **Underfitting** appears when both train and validation accuracy remain low (model too simple, too few epochs, too strong regularization, or too low LR).
- **Overfitting** appears when train accuracy becomes very high but validation lags or degrades (model too flexible relative to effective data, weak regularization, or very small training subset).


## 5) Objective (d): Regularization Techniques

We compare settings with weaker vs stronger regularization: dropout, weight decay, and augmentation.


In [ ]:
regularization_experiments = [
    {
        'name': 'weak_reg',
        'dropout': 0.0,
        'weight_decay': 0.0,
        'augment': False,
    },
    {
        'name': 'dropout_only',
        'dropout': 0.5,
        'weight_decay': 0.0,
        'augment': False,
    },
    {
        'name': 'dropout_wd_aug',
        'dropout': 0.5,
        'weight_decay': 1e-4,
        'augment': True,
    },
]

reg_histories = {}
reg_summary = []
for cfg in regularization_experiments:
    print(f"\n=== Regularization experiment: {cfg['name']} ===")
    tr_loader, va_loader, _ = get_dataloaders(
        DATA_ROOT,
        image_size=64,
        batch_size=32,
        num_workers=NUM_WORKERS,
        augment=cfg['augment'],
    )
    model = get_model('medium', num_classes=10, input_size=64, dropout=cfg['dropout'])
    history = train(
        model,
        tr_loader,
        va_loader,
        num_epochs=8,
        learning_rate=1e-3,
        optimizer_name='adam',
        weight_decay=cfg['weight_decay'],
        scheduler_name='cosine',
        device=DEVICE,
        verbose=False,
    )
    reg_histories[cfg['name']] = history
    reg_summary.append({
        **cfg,
        'best_val_acc': max(history['val_acc']),
        'final_train_acc': history['train_acc'][-1],
    })

reg_df = pd.DataFrame(reg_summary).sort_values('best_val_acc', ascending=False)
display(reg_df)
_ = compare_architectures(reg_histories, metric='val_acc', figsize=(9, 5))
plt.show()


### Objective (d) Interpretation
- Regularization generally improves generalization by reducing the train/validation gap.
- Dropout and weight decay constrain model co-adaptation.
- Data augmentation acts as data-space regularization and often gives robust gains.


## 6) Objective (e): Optimizer Comparison (Adam vs RMSprop vs SGD)

We keep architecture and most hyperparameters fixed, then compare optimizers.


In [ ]:
optimizer_histories = {}
optimizer_summary = []

for opt_name in ['adam', 'rmsprop', 'sgd']:
    print(f"\n=== Optimizer: {opt_name.upper()} ===")
    tr_loader, va_loader, _ = get_dataloaders(
        DATA_ROOT,
        image_size=64,
        batch_size=32,
        num_workers=NUM_WORKERS,
        augment=True,
    )
    model = get_model('deep', num_classes=10, input_size=64, dropout=0.5)
    history = train(
        model,
        tr_loader,
        va_loader,
        num_epochs=8,
        learning_rate=1e-3 if opt_name != 'sgd' else 3e-2,
        optimizer_name=opt_name,
        weight_decay=1e-4,
        scheduler_name='cosine',
        device=DEVICE,
        verbose=False,
    )
    optimizer_histories[opt_name] = history
    optimizer_summary.append({
        'optimizer': opt_name,
        'best_val_acc': max(history['val_acc']),
        'final_val_acc': history['val_acc'][-1],
        'final_train_acc': history['train_acc'][-1],
    })

opt_df = pd.DataFrame(optimizer_summary).sort_values('best_val_acc', ascending=False)
display(opt_df)
_ = compare_architectures(optimizer_histories, metric='val_acc', figsize=(9, 5))
plt.show()


### Objective (e) Interpretation
- Adam and RMSprop often converge faster in early epochs.
- SGD can match/beat adaptive methods with tuned learning rate and longer training.
- Optimizer performance is data/model dependent, so direct experiment-based comparison is essential.


## 7) Final Evaluation (Best Practical Configuration)

Use the best configuration from the experiments above, retrain with more epochs and (optionally) larger image size (128 or 224), then evaluate with confusion matrix and class report.


In [ ]:
# Example final configuration (replace with your best found settings)
FINAL_CFG = {
    'architecture': 'deep',
    'image_size': 128,
    'batch_size': 32,
    'dropout': 0.5,
    'num_epochs': 15,
    'learning_rate': 1e-3,
    'optimizer_name': 'adam',
    'weight_decay': 1e-4,
    'scheduler_name': 'cosine',
    'augment': True,
}

final_train_loader, final_val_loader, CLASS_NAMES = get_dataloaders(
    DATA_ROOT,
    image_size=FINAL_CFG['image_size'],
    batch_size=FINAL_CFG['batch_size'],
    num_workers=NUM_WORKERS,
    augment=FINAL_CFG['augment'],
)

final_model = get_model(
    FINAL_CFG['architecture'],
    num_classes=len(CLASS_NAMES),
    input_size=FINAL_CFG['image_size'],
    dropout=FINAL_CFG['dropout'],
)

final_history = train(
    final_model,
    final_train_loader,
    final_val_loader,
    num_epochs=FINAL_CFG['num_epochs'],
    learning_rate=FINAL_CFG['learning_rate'],
    optimizer_name=FINAL_CFG['optimizer_name'],
    weight_decay=FINAL_CFG['weight_decay'],
    scheduler_name=FINAL_CFG['scheduler_name'],
    device=DEVICE,
    verbose=True,
)

_ = plot_training_history(final_history, title='Final Model Training History')
plt.show()


In [ ]:
# Classification metrics and confusion matrix on validation split
y_true, y_pred = predict(final_model, final_val_loader, DEVICE)
print_classification_report(y_true, y_pred, CLASS_NAMES)
_ = plot_confusion_matrix(y_true, y_pred, CLASS_NAMES, title='Final Model Confusion Matrix')
plt.show()


In [ ]:
# Save final model
output_dir = REPO_ROOT / 'results'
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / 'best_cnn_model.pth'
torch.save(final_model.state_dict(), model_path)
print('Saved model to:', model_path)


## 8) Consolidated Objective Coverage Summary

- **(a) Progressive depth**: Section 2 compares Simple/Medium/Deep under fixed settings.
- **(b) Hyperparameter tuning**: Section 3 sweeps image size, batch size, learning rate, and dropout.
- **(c) Underfitting/overfitting**: Section 4 intentionally demonstrates and explains both failure modes.
- **(d) Regularization**: Section 5 compares dropout, weight decay, and augmentation combinations.
- **(e) Optimizers**: Section 6 compares Adam, RMSprop, and SGD using the same model backbone.

This single notebook contains the full investigation workflow end-to-end.
